# Task 6: Tableau Export

**Objective:** Export processed data, model metrics, and analytical results in formats suitable for Tableau visualization as specified in the assignment requirements.

This notebook focuses on exporting data and results for Tableau including model metrics tables, airline delay rates, airport delay rates, monthly delay trends, and training time statistics.

## Methodology

1. Load processed dataset and trained models from previous tasks
2. Export model performance metrics table
3. Export airline delay rates for Tableau visualization
4. Export airport delay rates (origin and destination)
5. Export monthly delay trends
6. Export training time statistics
7. Provide guidance on importing these files into Tableau
8. Provide interpretation notes for academic report

In [ ]:
# Load and prepare dataset (same procedure as previous tasks)
DATA_PATH = "C:\\Users\\shiva\\Downloads\\On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1\\On_Time_Reporting_Carrier_On_Time_Performance_2024_1.csv"
print("Loading and preparing dataset...")

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "") \
    .option("treatEmptyValuesAsNulls", "true") \
    .csv(DATA_PATH)

# Apply data preparation steps
df_clean = df.dropDuplicates()

# Handle missing values
numerical_cols = [field.name for field in df_clean.schema.fields if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType))]
string_cols = [field.name for field in df_clean.schema.fields if isinstance(field.dataType, StringType)]

df_filled = df_clean
for col_name in numerical_cols:
    if col_name in df_clean.columns:
        median_val = df_clean.approxQuantile(col_name, [0.5], 0.25)[0]
        if median_val is not None:
            df_filled = df_filled.fillna({col_name: median_val})
        else:
            df_filled = df_filled.fillna({col_name: 0})

for col_name in string_cols:
    if col_name in df_clean.columns:
        df_filled = df_filled.fillna({col_name: "Unknown"})

# Remove cancelled and diverted flights
df_filtered = df_filled.filter((col("Cancelled") == 0) & (col("Diverted") == 0))

# Remove leakage variables and identifiers
leakage_vars = ["ArrDelay", "ArrDelayMinutes", "ArrivalDelayGroups", "ArrTime", "ArrTimeBlk"]
identifier_cols = ["Tail_Number", "Flight_Number_Reporting_Airline", "FlightDate"]
cols_to_remove = [c for c in leakage_vars + identifier_cols if c in df_filtered.columns]
df_clean = df_filtered.drop(*cols_to_remove)

# Select features
numeric_features = ["Month", "DayOfWeek", "CRSDepTime", "DepDelay", "TaxiOut", "Distance", "AirTime"]
categorical_features = ["Reporting_Airline", "Origin", "Dest"]
label_column = "ArrDel15"
required_columns = numeric_features + categorical_features + [label_column]
df_selected = df_clean.select(required_columns)

# Apply sampling
df_sampled = df_selected.sampleBy(
    label_column,
    fractions={0: 0.15, 1: 0.15},
    seed=42
)

# Split into training and test sets
train_data, test_data = df_sampled.randomSplit([0.8, 0.2], seed=42)
train_data.cache()
test_data.cache()

print(f"Dataset ready: {df_sampled.count():,} rows for modeling")
print(f"Training set: {train_data.count():,} rows")
print(f"Test set: {test_data.count():,} rows")

## Helper Functions

Define functions for model training and evaluation needed for exports.

In [ ]:
# Define feature sets
numeric_features = ["Month", "DayOfWeek", "CRSDepTime", "DepDelay", "TaxiOut", "Distance", "AirTime"]
categorical_features = ["Reporting_Airline", "Origin", "Dest"]
label_column = "ArrDel15"

def create_pipeline(classifier):
    """Create a reusable PySpark ML Pipeline"""
    # StringIndexer for categorical features
    string_indexers = [
        StringIndexer(inputCol=cat, outputCol=f"{cat}_indexed")
        for cat in categorical_features
    ]

    # OneHotEncoder for categorical variables
    one_hot_encoders = [
        OneHotEncoder(inputCol=f"{cat}_indexed", outputCol=f"{cat}_encoded")
        for cat in categorical_features
    ]

    # VectorAssembler to combine all features
    indexed_categorical_cols = [f"{cat}_indexed" for cat in categorical_features]
    encoded_categorical_cols = [f"{cat}_encoded" for cat in categorical_features]
    all_features = numeric_features + encoded_categorical_cols
    assembler = VectorAssembler(inputCols=all_features, outputCol="features")

    # StandardScaler for feature normalization
    scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)

    # Create pipeline stages
    stages = string_indexers + one_hot_encoders + [assembler, scaler, classifier]
    
    return Pipeline(stages=stages)

def get_param_grids():
    """Define parameter grids for hyperparameter tuning"""
    param_grids = {
        "Logistic Regression": ParamGridBuilder() \
            .addGrid(LogisticRegression().regParam, [0.01, 0.1]) \
            .addGrid(LogisticRegression().elasticNetParam, [0.0, 0.5]) \
            .build(),
        
        "Decision Tree": ParamGridBuilder() \
            .addGrid(DecisionTreeClassifier().maxDepth, [5, 10]) \
            .addGrid(DecisionTreeClassifier().maxBins, [32, 64]) \
            .build(),
        
        "Random Forest": ParamGridBuilder() \
            .addGrid(RandomForestClassifier().numTrees, [50, 100]) \
            .addGrid(RandomForestClassifier().maxDepth, [8, 12]) \
            .build(),
    
        "GBT Classifier": ParamGridBuilder() \
            .addGrid(GBTClassifier().maxIter, [20, 50]) \
            .addGrid(GBTClassifier().maxDepth, [5, 8]) \
            .build()
    }
    
    return param_grids

def evaluate_model(model, test_data, label_col):
    """Evaluate a trained model and return performance metrics"""
    # Make predictions
    predictions = model.transform(test_data)

    # Initialize evaluators
    binary_evaluator = BinaryClassificationEvaluator(
        labelCol=label_col, 
    
        rawPredictionCol="rawPrediction", 
        metricName="areaUnderROC"
    )
    
    multi_evaluator = MulticlassClassificationEvaluator(
        labelCol=label_col, 
        predictionCol="prediction"
    )
    
    # Calculate metrics
    auc = binary_evaluator.evaluate(predictions)
    accuracy = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "accuracy"})
    precision = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "weightedPrecision"})
    recall = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "weightedRecall"})
    f1 = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "weightedFMeasure"})
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

In [ ]:
# Define classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(featuresCol="scaledFeatures", labelCol=label_column, maxIter=100),
    "Decision Tree": DecisionTreeClassifier(featuresCol="scaledFeatures", labelCol=label_column),
    "Random Forest": RandomForestClassifier(featuresCol="scaledFeatures", labelCol=label_column),
    "GBT Classifier": GBTClassifier(featuresCol="scaledFeatures", labelCol=label_column)
}

# Get parameter grids
param_grids = get_param_grids()

# Dictionary to store results
model_results = {}
trained_models = {}

# Train and evaluate each model
print("\\nTraining models for export...")
for model_name, classifier in classifiers.items():
    print(f"\\nTraining {model_name}...")

    start_time = time.time()

    # Create pipeline
    pipeline = create_pipeline(classifier)

    # Set up cross-validation
    crossval = CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=param_grids[model_name],
        evaluator=BinaryClassificationEvaluator(labelCol=label_column, rawPredictionCol="rawPrediction", metricName="areaUnderROC"),

        numFolds=3,
        seed=42
    )

    # Train model
    cv_model = crossval.fit(train_data)
    
    training_time = time.time() - start_time
    
    # Get best model
    best_model = cv_model.bestModel
    
    # Evaluate model
    metrics = evaluate_model(best_model, test_data, label_column)
    
    metrics["training_time"] = training_time
    
    # Store results
    model_results[model_name] = metrics
    trained_models[model_name] = best_model
    
    # Print results
    print(f"Results for {model_name}:")
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1-Score: {metrics['f1']:.4f}")
    print(f"  AUC-ROC: {metrics['auc']:.4f}")
    print(f"  Training Time: {training_time:.2f} seconds")
    
    # Clean up
    del cv_model

In [ ]:
# ============================================================================
# EXPORT 1: MODEL PERFORMANCE METRICS TABLE
# ============================================================================

print("\\n=== EXPORTING MODEL PERFORMANCE METRICS TABLE ===")

# Create DataFrame from model results
metrics_data = []
for model_name, metrics in model_results.items():
    metrics_data.append({
        "Model": model_name,
        "Accuracy": round(metrics['accuracy'], 4),
        "Precision": round(metrics['precision'], 4),
        "Recall": round(metrics['recall'], 4),
        "F1_Score": round(metrics['f1'], 4),
        "AUC_ROC": round(metrics['auc'], 4),
        "Training_Time_Seconds": round(metrics['training_time'], 2)
    })

# Create DataFrame
metrics_df = spark.createDataFrame(metrics_data)

# Show the metrics table
print("Model Performance Metrics for Tableau:")
metrics_df.show(truncate=False)

# Export to CSV for Tableau
metrics_path = f"{TABLEAU_DIR}/model_performance_metrics.csv"
metrics_df.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(metrics_path)

print(f"Model performance metrics exported to: {metrics_path}")

In [ ]:
# ============================================================================
# EXPORT 2: AIRLINE DELAY RATES
# ============================================================================

print("\\n=== EXPORTING AIRLINE DELAY RATES ===")

# Calculate delay rates by airline
airline_delay_stats = df_selected.groupBy("Reporting_Airline").agg(
    count("*").alias("Total_Flights"),
    sum(col(label_column)).alias("Delayed_Flights"),
    (sum(col(label_column)) / count("*") * 100).alias("Delay_Rate_Percentage")
).orderBy(desc("Delay_Rate_Percentage"))

print("Airline Delay Rates (Top 10):")
airline_delay_stats.show(10, truncate=False)

# Export to CSV for Tableau
airline_path = f"{TABLEAU_DIR}/airline_delay_rates.csv"
airline_delay_stats.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(airline_path)

print(f"Airline delay rates exported to: {airline_path}")

# Also export origin airport delay rates
print("\\n=== EXPORTING ORIGIN AIRPORT DELAY RATES ===")
origin_delay_stats = df_selected.groupBy("Origin").agg(
    count("*").alias("Total_Flights"),
    sum(col(label_column)).alias("Delayed_Flights"),
    (sum(col(label_column)) / count("*") * 100).alias("Delay_Rate_Percentage")
).orderBy(desc("Delay_Rate_Percentage"))

print("Origin Airport Delay Rates (Top 10):")
origin_delay_stats.show(10, truncate=False)

origin_path = f"{TABLEAU_DIR}/origin_airport_delay_rates.csv"
origin_delay_stats.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(origin_path)

print(f"Origin airport delay rates exported to: {origin_path}")

# Also export destination airport delay rates
print("\\n=== EXPORTING DESTINATION AIRPORT DELAY RATES ===")
dest_delay_stats = df_selected.groupBy("Dest").agg(
    count("*").alias("Total_Flights"),
    sum(col(label_column)).alias("Delayed_Flights"),
    (sum(col(label_column)) / count("*") * 100).alias("Delay_Rate_Percentage")
).orderBy(desc("Delay_Rate_Percentage"))

print("Destination Airport Delay Rates (Top 10):")
dest_delay_stats.show(10, truncate=False)

dest_path = f"{TABLEAU_DIR}/destination_airport_delay_rates.csv"
dest_delay_stats.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(dest_path)

print(f"Destination airport delay rates exported to: {dest_path}")

In [ ]:
# ============================================================================
# EXPORT 3: MONTHLY DELAY TRENDS
# ============================================================================

print("\\n=== EXPORTING MONTHLY DELAY TRENDS ===")

# Calculate delay trends by month
monthly_delay_stats = df_selected.groupBy("Month").agg(
    count("*").alias("Total_Flights"),
    sum(col(label_column)).alias("Delayed_Flights"),
    (sum(col(label_column)) / count("*") * 100).alias("Delay_Rate_Percentage")
).orderBy("Month")

print("Monthly Delay Trends:")
monthly_delay_stats.show(12, truncate=False)

# Export to CSV for Tableau
monthly_path = f"{TABLEAU_DIR}/monthly_delay_trends.csv"
monthly_delay_stats.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(monthly_path)

print(f"Monthly delay trends exported to: {monthly_path}")

# Also create a version with month names for better Tableau visualization
month_names = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December"
}

# Add month names
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

get_month_name = udf(lambda x: month_names.get(x, "Unknown"), StringType())
monthly_with_names = monthly_delay_stats.withColumn("Month_Name", get_month_name(col("Month")))

print("\\nMonthly Delay Trends with Month Names:")
monthly_with_names.show(12, truncate=False)

monthly_named_path = f"{TABLEAU_DIR}/monthly_delay_trends_with_names.csv"
monthly_with_names.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(monthly_named_path)

print(f"Monthly delay trends with names exported to: {monthly_named_path}")

In [ ]:
# ============================================================================
# EXPORT 4: TRAINING TIME STATISTICS
# ============================================================================

print("\\n=== EXPORTING TRAINING TIME STATISTICS ===")

# Create training time statistics DataFrame
training_time_data = []
for model_name, metrics in model_results.items():
    training_time_data.append({
        "Model": model_name,
        "Training_Time_Seconds": round(metrics['training_time'], 2),
        "Training_Time_Minutes": round(metrics['training_time'] / 60, 2),
        "Relative_Performance": round(metrics['auc'], 4)  # Using AUC as performance metric
    })

training_time_df = spark.createDataFrame(training_time_data)

print("Training Time Statistics:")
training_time_df.show(truncate=False)

# Export to CSV for Tableau
training_time_path = f"{TABLEAU_DIR}/training_time_statistics.csv"
training_time_df.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(training_time_path)

print(f"Training time statistics exported to: {training_time_path}")

# ============================================================================
# TABLEAU IMPORT GUIDANCE
# ============================================================================

print("\\n" + "="*60)
print("TABLEAU IMPORT GUIDANCE")
print("="*60)

print("\\nTo import these files into Tableau:")
print("1. Open Tableau Desktop")
print("2. Select 'Connect to Data' -> 'Text File'")
print("3. Navigate to the tableau_exports directory")
print("4. Select the CSV files you want to visualize:")
print("   - model_performance_metrics.csv")
print("   - airline_delay_rates.csv")
print("   - origin_airport_delay_rates.csv")
print("   - destination_airport_delay_rates.csv")
print("   - monthly_delay_trends.csv (or monthly_delay_trends_with_names.csv)")
print("   - training_time_statistics.csv")
print("5. Drag each file to the data source pane")
print("6. Use Tableau's join/union features to combine related data as needed")
print("7. Create worksheets, dashboards, and stories for your analysis")

print("\\nRecommended Tableau Visualizations:")
print("1. Model Performance Comparison: Bar chart comparing Accuracy, Precision, Recall, F1, AUC across models")
print("2. Airline Delay Rates: Bar chart showing top 10 airlines by delay rate")
print("3. Airport Delay Patterns: Heat map or bar chart showing delay rates by origin/destination")
print("4. Monthly Trends: Line chart showing delay rates by month")
print("5. Training Efficiency: Scatter plot of training time vs. model performance (AUC)")

print("\\n" + "="*60)
print("SUMMARY OF EXPORTED FILES")
print("="*60)

import os
if os.path.exists(TABLEAU_DIR):
    files = os.listdir(TABLEAU_DIR)
    print(f"\\nExported files in {TABLEAU_DIR}:")
    for file in sorted(files):
        file_path = os.path.join(TABLEAU_DIR, file)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path)
            print(f"  {file} ({size} bytes)")
else:
    print(f"\\nDirectory {TABLEAU_DIR} does not exist.")

print("\\n" + "="*60)
print("TASK 6 COMPLETE: TABLEAU EXPORT FINISHED")
print("="*60)

# Stop Spark session
spark.stop()
print("\\nSpark session stopped.")